# 00.2 - Big-O Notation & Complexity Analysis

Big-O answers: **"How does the running time (or memory) grow when the input gets bigger?"**

We don't measure seconds (they depend on the computer). We count **how the number of steps grows with n** (the input size).

## Why it matters
- Interviewers **always** ask: *"What's the time and space complexity?"*
- LeetCode gives constraints like `1 <= n <= 10^5` - that tells you which complexity is fast enough.

## 1. The common complexities (best → worst)

| Big-O | Name | Example | n = 1,000,000 → steps |
|---|---|---|---|
| **O(1)** | constant | `a[i]`, `d[key]`, `x in set` | 1 |
| **O(log n)** | logarithmic | binary search, heap push/pop | ~20 |
| **O(n)** | linear | one loop over the data | 1,000,000 |
| **O(n log n)** | linearithmic | sorting (merge sort, Python `sorted`) | ~20,000,000 |
| **O(n²)** | quadratic | two nested loops | 10¹² |
| **O(2ⁿ)** | exponential | all subsets, naive recursion | |
| **O(n!)** | factorial | all permutations | |

### What fits in ~1 second (≈ 10⁷–10⁸ simple operations in Python… realistically 10⁷)
| Constraint on n | Complexity you need |
|---|---|
| n ≤ 10 | O(n!) okay |
| n ≤ 20 | O(2ⁿ) |
| n ≤ 500 | O(n³) |
| n ≤ 5,000 | O(n²) |
| n ≤ 10⁶ | O(n log n) or O(n) |
| n ≥ 10⁸ | O(log n) or O(1) |

In [1]:
import math
sizes = [10, 100, 1_000, 10_000, 100_000]
print(f"{'n':>8} {'log n':>7} {'n':>9} {'n log n':>11} {'n^2':>15} {'2^n':>12}")
for n in sizes:
    two_n = f"{2**n:.1e}" if n <= 1000 else "huge"
    print(f"{n:>8} {math.log2(n):>7.1f} {n:>9} {int(n*math.log2(n)):>11} {n*n:>15} {two_n:>12}")

       n   log n         n     n log n             n^2          2^n
      10     3.3        10          33             100      1.0e+03
     100     6.6       100         664           10000      1.3e+30
    1000    10.0      1000        9965         1000000     1.1e+301
   10000    13.3     10000      132877       100000000         huge
  100000    16.6    100000     1660964     10000000000         huge


## 2. Examples of each complexity

In [2]:
# O(1) - constant: doesn't depend on n
def get_first(items):
    return items[0]

# O(log n) - halve the problem each step
def count_halvings(n):
    steps = 0
    while n > 1:
        n //= 2
        steps += 1
    return steps

# O(n) - touch every element once
def find_max(items):
    best = items[0]
    for x in items:
        best = max(best, x)
    return best

# O(n log n) - sorting
def sort_items(items):
    return sorted(items)

# O(n^2) - every pair
def has_pair_with_sum_slow(items, target):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] + items[j] == target:
                return True
    return False

# O(2^n) - every subset
def all_subsets(items):
    if not items:
        return [[]]
    rest = all_subsets(items[1:])
    return rest + [[items[0]] + s for s in rest]

print(get_first([7, 8]), count_halvings(1_000_000), find_max([3, 9, 2]),
      sort_items([3, 1, 2]), has_pair_with_sum_slow([1, 4, 6], 10), len(all_subsets([1, 2, 3, 4])))

7 19 9 [1, 2, 3] True 16


## 3. Seeing it: measure real running times
Double n and watch what happens: O(n) doubles, O(n²) quadruples.

In [3]:
import time

def linear(n):
    total = 0
    for i in range(n):
        total += i
    return total

def quadratic(n):
    total = 0
    for i in range(n):
        for j in range(n):
            total += 1
    return total

def timeit(f, n):
    start = time.perf_counter()
    f(n)
    return time.perf_counter() - start

for n in [500, 1000, 2000]:
    print(f"n={n:>5}  O(n): {timeit(linear, n)*1000:7.3f} ms   O(n^2): {timeit(quadratic, n)*1000:8.1f} ms")

n=  500  O(n):   0.016 ms   O(n^2):     11.2 ms
n= 1000  O(n):   0.028 ms   O(n^2):     38.6 ms


n= 2000  O(n):   0.072 ms   O(n^2):    190.9 ms

## 4. Rules for calculating Big-O

1. **Drop constants:** O(2n) → O(n), O(500) → O(1).
2. **Drop smaller terms:** O(n² + n) → O(n²).
3. **Sequential steps add:** loop over a, then loop over b → O(a + b).
4. **Nested steps multiply:** loop over a, inside it loop over b → O(a × b).
5. **Different inputs, different letters:** two lists of sizes n and m → O(n + m), not O(n).
6. **Halving each step → log n.**
7. **Recursion:** (number of calls) × (work per call). Branching factor b, depth d → O(bᵈ).

In [4]:
def example(a, b):
    total = 0
    for x in a:          # O(n)
        total += x
    for x in a:          # O(n)   -> still O(n) total (2n -> n)
        for y in b:      # O(m)   -> nested: O(n * m)
            total += x * y
    return total
# Total: O(n + n*m) = O(n*m)

def halving(n):
    while n > 0:         # n, n/2, n/4, ... -> O(log n)
        n //= 2

def nested_dependent(n):
    count = 0
    for i in range(n):
        for j in range(i):   # 0 + 1 + 2 + ... + (n-1) = n(n-1)/2 -> O(n^2)
            count += 1
    return count

print(example([1, 2], [3, 4]), nested_dependent(10))

24 45


## 5. Space complexity
How much **extra memory** grows with n (input itself usually not counted).

| Code | Extra space |
|---|---|
| a few variables | O(1) |
| new list / dict / set of size n | O(n) |
| 2D grid n × n | O(n²) |
| recursion depth d | O(d) - each call uses stack memory |

In [5]:
def reverse_in_place(a):          # O(1) extra space
    l, r = 0, len(a) - 1
    while l < r:
        a[l], a[r] = a[r], a[l]
        l += 1; r -= 1
    return a

def reverse_copy(a):              # O(n) extra space
    return a[::-1]

def recursive_sum(a, i=0):        # O(n) space: n calls on the call stack
    if i == len(a):
        return 0
    return a[i] + recursive_sum(a, i + 1)

print(reverse_in_place([1, 2, 3]), reverse_copy([1, 2, 3]), recursive_sum([1, 2, 3]))

[3, 2, 1] [3, 2, 1] 6


## 6. Best, average, worst case & amortized

- **Worst case** (what interviews usually mean): linear search when the item is last → O(n).
- **Best case:** item is first → O(1). Rarely useful.
- **Average case:** quicksort is O(n log n) on average, O(n²) worst.
- **Amortized:** `list.append` is O(1) *on average* - occasionally Python resizes the array (O(n)), but that's rare enough that the average stays O(1).

In [6]:
import sys
a = []
last = sys.getsizeof(a)
print("len -> bytes allocated (watch the jumps = resizes)")
for i in range(40):
    a.append(i)
    size = sys.getsizeof(a)
    if size != last:
        print(f"{len(a):>3} -> {size}")
        last = size

len -> bytes allocated (watch the jumps = resizes)
  1 -> 88
  5 -> 120
  9 -> 184
 17 -> 248
 25 -> 312
 33 -> 376


## 7. Time complexity of Python built-ins (memorise this)

| Operation | list | dict / set | deque |
|---|---|---|---|
| index `a[i]` | O(1) | - | O(n) |
| append / add | O(1)* | O(1)* | O(1) both ends |
| pop from end | O(1) | - | O(1) |
| pop(0) / insert(0, x) | **O(n)** | - | **O(1)** |
| `x in ...` | **O(n)** | **O(1)*** | O(n) |
| delete by key / value | O(n) | O(1)* | O(n) |
| sort | O(n log n) | - | - |
| slice `a[i:j]` | O(j−i) | - | - |
| `len()` | O(1) | O(1) | O(1) |

\* average / amortized. Strings: concatenation in a loop is O(n²) total → use `"".join(list)`.

| Other | Complexity |
|---|---|
| `heapq.heappush / heappop` | O(log n) |
| `heapq.heapify` | O(n) |
| `bisect.bisect_left` | O(log n) (`insort` is O(n) due to shifting) |
| `sorted(), list.sort()` | O(n log n) |
| `Counter(iterable)` | O(n) |
| `min(), max(), sum()` | O(n) |

## 8. Practice: what is the complexity?
Try to answer before revealing (answers in the next cell).

In [7]:
def q1(n):
    for i in range(n):
        for j in range(10):
            pass

def q2(n):
    i = 1
    while i < n:
        i *= 2

def q3(items):
    items.sort()
    for x in items:
        pass

def q4(n):
    for i in range(n):
        j = 1
        while j < n:
            j *= 2

def q5(n):
    if n <= 1:
        return 1
    return q5(n - 1) + q5(n - 1)

In [8]:
answers = {
    "q1": "O(n)        - inner loop is constant (10)",
    "q2": "O(log n)    - i doubles each step",
    "q3": "O(n log n)  - sort dominates the O(n) loop",
    "q4": "O(n log n)  - n times an O(log n) loop",
    "q5": "O(2^n)      - each call makes 2 calls, depth n",
}
for k, v in answers.items():
    print(k, "->", v)

q1 -> O(n)        - inner loop is constant (10)
q2 -> O(log n)    - i doubles each step
q3 -> O(n log n)  - sort dominates the O(n) loop
q4 -> O(n log n)  - n times an O(log n) loop
q5 -> O(2^n)      - each call makes 2 calls, depth n


## Key takeaways
- Big-O describes **growth**, not exact time.
- Drop constants and smaller terms; nested loops multiply.
- Know the cost of Python built-ins - `x in list` is O(n), `x in set` is O(1).
- Always state **both time and space** in interviews.

**Next: `00.3-Python-Toolkit-for-DSA.ipynb`**